In [ ]:
# --- Setup: installs and imports (run once per session) ---
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install -U "opencv-python>=4.10.0.84" mediapipe ultralytics pandas scikit-image matplotlib


import os, json, math, io, base64, glob, shutil, time, uuid
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

import torch
import mediapipe as mp
from ultralytics import YOLO
from skimage import morphology, measure

try:
    from google.colab import files, output
except Exception:
    files = None
    output = None

print("Setup complete.")

Found existing installation: opencv-python 4.13.0.90
Uninstalling opencv-python-4.13.0.90:
  Successfully uninstalled opencv-python-4.13.0.90
Found existing installation: opencv-contrib-python 4.13.0.90
Uninstalling opencv-contrib-python-4.13.0.90:
  Successfully uninstalled opencv-contrib-python-4.13.0.90
  Using cached opencv_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached mediapipe-0.10.32-py3-none-manylinux_2_28_x86_64.whl.metadata (9.8 kB)
  Using cached opencv_contrib_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
Using cached mediapipe-0.10.32-py3-none-manylinux_2_28_x86_64.whl (10.3 MB)
Using cached opencv_contrib_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl (79.2 MB)
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.14
    Uninstalling mediapipe-0.10.14:
      Successfully uninstalled mediapipe-0.

In [ ]:
# --- Configuration & directories ---
PRODUCT_NAME = "demo_product_001"   # change per item
BASE_DIR = Path("/content")
SESSION_DIR = BASE_DIR / f"scan_{PRODUCT_NAME}"
IMAGES_DIR  = SESSION_DIR / "images"
OVERLAY_DIR = SESSION_DIR / "overlays"
EXPORT_DIR  = SESSION_DIR / "export"
REPORTS_DIR = SESSION_DIR / "reports"
YOLO_MODEL_DIR = SESSION_DIR / "yolo_models"  # optional

for d in [IMAGES_DIR, OVERLAY_DIR, EXPORT_DIR, REPORTS_DIR, YOLO_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Damage classes we’ll track
DAMAGE_CLASSES = ["hole", "scratch", "stain", "fray_edge", "hinge_screw_issue"]

# Quality gate thresholds
BLUR_THRESHOLD = 120.0   # variance of Laplacian; raise if lighting is good (80–200 typical)
HAND_IOU_THRESHOLD = 0.02  # fraction of object area occluded by hand

print("Dirs:", SESSION_DIR)

Dirs: /content/scan_demo_product_001


In [ ]:
# --- Utils ---
def save_image(np_img, out_path): cv2.imwrite(str(out_path), np_img)
def read_image(path): return cv2.imread(str(path))
def list_images(folder=None):
    folder = IMAGES_DIR if folder is None else folder
    paths = []
    for e in ("*.jpg","*.jpeg","*.png"): paths += glob.glob(str(folder / e))
    return sorted(paths)
print("Utils loaded.")

Utils loaded.


In [ ]:
def upload_images():
    if files is None:
        print("Colab 'files' not available.")
        return
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(IMAGES_DIR / name, "wb") as f:
            f.write(content)
    print(f"Uploaded {len(uploaded)} images to {IMAGES_DIR}")

upload_images()


Saving Back.jpg to Back.jpg
Uploaded 1 images to /content/scan_demo_product_001/images


In [ ]:
from mediapipe.python.solutions import hands as mp_hands

def variance_of_laplacian(gray):
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def quick_object_mask(img):
    # GrabCut from central rectangle; fallback to Otsu
    h, w = img.shape[:2]
    rect = (int(0.1*w), int(0.1*h), int(0.8*w), int(0.8*h))
    mask = np.zeros((h, w), np.uint8)
    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(img, mask, rect, bgdModel, fgdModel, 3, cv2.GC_INIT_WITH_RECT)
        mask2 = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')
    except:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        v = hsv[:,:,2]
        thr, bw = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        mask2 = np.zeros_like(v, dtype=np.uint8)
        if contours:
            c = max(contours, key=cv2.contourArea)
            cv2.drawContours(mask2, [c], -1, 1, thickness=cv2.FILLED)
    return mask2

def hand_overlap_fraction(img, obj_mask):
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2) as hands:
        results = hands.process(img_rgb)
    if not results.multi_hand_landmarks:
        return 0.0
    hand_mask = np.zeros((h, w), dtype=np.uint8)
    for handLms in results.multi_hand_landmarks:
        for lm in handLms.landmark:
            x = int(lm.x * w); y = int(lm.y * h)
            cv2.circle(hand_mask, (x,y), 20, 1, -1)
    inter = np.logical_and(hand_mask>0, obj_mask>0).sum()
    obj_area = (obj_mask>0).sum() + 1e-6
    return inter / obj_area

def quality_gate(path, blur_th=BLUR_THRESHOLD, iou_th=HAND_IOU_THRESHOLD, save_overlay=True):
    img = read_image(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    vblur = variance_of_laplacian(gray)
    obj_mask = quick_object_mask(img)
    frac = hand_overlap_fraction(img, obj_mask)
    status = {"path": str(path), "var_laplacian": float(vblur), "hand_obj_overlap": float(frac),
              "pass": bool((vblur >= blur_th) and (frac <= iou_th))}
    if save_overlay:
        overlay = img.copy()
        overlay[obj_mask==0] = (overlay[obj_mask==0] * 0.3).astype(np.uint8)
        cv2.putText(overlay, f"BlurVar={vblur:.1f} HandOverlap={frac:.3f} PASS={status['pass']}",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0,255,0) if status["pass"] else (0,0,255), 2)
        out = OVERLAY_DIR / (Path(path).stem + "_quality.jpg")
        cv2.imwrite(str(out), overlay)
        status["quality_overlay"] = str(out)
    return status

def run_quality_gate_on_all():
    rows = []
    for p in list_images():
        rows.append(quality_gate(p))
    df = pd.DataFrame(rows)
    fp = EXPORT_DIR / "quality_gate.csv"
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return df

print("Quality gate ready.")

Quality gate ready.


In [ ]:
# --- Heuristic detectors for: scratches, stains, holes, fray/edge; hinge/screw placeholder ---

def detect_scratches(img, obj_mask):
    # thin elongated edges on object
    work = cv2.bilateralFilter(img, 5, 50, 50)
    gray = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 60, 150)
    edges[obj_mask==0] = 0
    density = morphology.binary_dilation(edges>0, morphology.disk(1)).astype(np.uint8)*255
    num, lab = cv2.connectedComponents(density)
    bboxes = []
    for k in range(1, num):
        ys, xs = np.where(lab==k)
        if len(xs) < 50: continue
        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()
        h = y1 - y0 + 1; w = x1 - x0 + 1
        if max(h, w) / (min(h, w) + 1e-6) < 2.5: continue  # elongated
        bboxes.append((x0,y0,x1,y1, float(min(1.0, len(xs)/500.0)) ))
    return bboxes

def detect_stains(img, obj_mask):
    # Lab color outliers inside object region
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    A, B = lab[:,:,1], lab[:,:,2]
    valid = (obj_mask>0)
    if valid.sum() < 10: return []
    mA, sA = A[valid].mean(), A[valid].std()+1e-6
    mB, sB = B[valid].mean(), B[valid].std()+1e-6
    z = ((A-mA)/sA)**2 + ((B-mB)/sB)**2
    z[~valid] = 0
    thr = np.percentile(z[valid], 95)
    cand = (z>thr).astype(np.uint8)
    cand = morphology.remove_small_holes(cand.astype(bool), 64)
    cand = morphology.remove_small_objects(cand, 64)
    cand = morphology.binary_dilation(cand, morphology.disk(3)).astype(np.uint8)
    lab_img = measure.label(cand, connectivity=2)
    bboxes = []
    for r in measure.regionprops(lab_img):
        y0,x0,y1,x1 = r.bbox
        area = r.area
        conf = min(1.0, area/2000.0)
        bboxes.append((x0,y0,x1-1,y1-1,float(conf)))
    return bboxes

def detect_holes(img, obj_mask):
    # dark, roughly roundish blobs (snags/holes)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    inv = 255 - gray
    inv[obj_mask==0] = 0
    thr = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
    thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((3,3), np.uint8), iterations=1)
    lab = measure.label(thr>0, connectivity=2)
    bboxes = []
    for r in measure.regionprops(lab):
        y0,x0,y1,x1 = r.bbox
        area = r.area
        if area < 50: continue
        per = r.perimeter + 1e-6
        circ = 4*math.pi*area/(per*per)
        if circ < 0.2: continue
        conf = min(1.0, area/1500.0)
        bboxes.append((x0,y0,x1-1,y1-1, float(conf)))
    return bboxes

def detect_fray_edge(img, obj_mask):
    # edge wear near object boundary band
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 120)
    band = morphology.binary_dilation(obj_mask>0, morphology.disk(5)) ^ (obj_mask>0)
    band = band.astype(np.uint8)
    cand = (edges>0) & (band>0)
    lab = measure.label(cand, connectivity=2)
    bboxes = []
    for r in measure.regionprops(lab):
        y0,x0,y1,x1 = r.bbox
        area = r.area
        if area < 80: continue
        conf = min(1.0, area/3000.0)
        bboxes.append((x0,y0,x1-1,y1-1,float(conf)))
    return bboxes

def detect_hinge_screw_issue(img, obj_mask):
    # placeholder (use YOLO supervised head when you have labels)
    return []

print("Heuristic detectors ready.")

Heuristic detectors ready.


In [ ]:
YOLO_ENABLE = False     # set True when you have a labeled dataset + YAML
YOLO_MODEL_NAME = str(YOLO_MODEL_DIR / "yolov8n_damage.pt")
YOLO_CLASSES = DAMAGE_CLASSES

def yolo_train_if_enabled(data_yaml_path, epochs=50, imgsz=640):
    if not YOLO_ENABLE:
        print("YOLO disabled.")
        return None
    model = YOLO("yolov8n.pt")
    model.train(data=data_yaml_path, epochs=epochs, imgsz=imgsz, project=str(YOLO_MODEL_DIR), name="exp", exist_ok=True)
    ckpts = sorted(glob.glob(str(YOLO_MODEL_DIR / "exp" / "weights" / "best.pt")))
    if ckpts:
        shutil.copy(ckpts[-1], YOLO_MODEL_NAME)
        print("Saved best model to", YOLO_MODEL_NAME)
    return YOLO_MODEL_NAME

def yolo_infer_if_enabled(img):
    if not YOLO_ENABLE or not Path(YOLO_MODEL_NAME).exists():
        return []
    model = YOLO(YOLO_MODEL_NAME)
    res = model.predict(img, verbose=False, conf=0.3, iou=0.5)
    bboxes = []
    for r in res:
        for b in r.boxes:
            x0,y0,x1,y1 = b.xyxy[0].tolist()
            conf = float(b.conf[0].item())
            cls = int(b.cls[0].item())
            label = YOLO_CLASSES[cls] if cls < len(YOLO_CLASSES) else "damage"
            bboxes.append((int(x0),int(y0),int(x1),int(y1),conf,label))
    return bboxes

print("YOLO head (optional) ready.")

YOLO head (optional) ready.


In [ ]:
def run_detectors_on_image(path):
    img = read_image(path)
    obj_mask = quick_object_mask(img)

    results = []

    # Heuristics
    for (label, func) in [
        ("scratch", detect_scratches),
        ("stain", detect_stains),
        ("hole", detect_holes),
        ("fray_edge", detect_fray_edge),
        ("hinge_screw_issue", detect_hinge_screw_issue),
    ]:
        boxes = func(img, obj_mask)
        for (x0,y0,x1,y1,conf) in boxes:
            results.append({"type": label, "bbox": [int(x0),int(y0),int(x1),int(y1)], "confidence": float(conf), "source": "heuristic"})

    # Optional YOLO
    for (x0,y0,x1,y1,conf,label) in yolo_infer_if_enabled(img):
        results.append({"type": label, "bbox": [x0,y0,x1,y1], "confidence": conf, "source": "yolo"})

    # Overlay
    overlay = img.copy()
    cmap = {
        "hole": (255, 0, 0),
        "scratch": (0, 255, 0),
        "stain": (0, 0, 255),
        "fray_edge": (255, 255, 0),
        "hinge_screw_issue": (255, 0, 255)
    }
    for r in results:
        x0,y0,x1,y1 = r["bbox"]
        cv2.rectangle(overlay, (x0,y0), (x1,y1), cmap.get(r["type"],(255,255,255)), 2)
        cv2.putText(overlay, f"{r['type']}:{r['confidence']:.2f}", (x0, max(0,y0-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, cmap.get(r["type"],(255,255,255)), 1)

    out_path = OVERLAY_DIR / (Path(path).stem + "_overlay.jpg")
    cv2.imwrite(str(out_path), overlay)

    return results, str(out_path)

def run_batch_detection():
    rows = []
    json_records = {}
    for p in list_images():
        rlist, overlay_path = run_detectors_on_image(p)
        for r in rlist:
            rows.append({
                "image": str(p),
                "type": r["type"],
                "x0": r["bbox"][0], "y0": r["bbox"][1],
                "x1": r["bbox"][2], "y1": r["bbox"][3],
                "confidence": r["confidence"],
                "source": r["source"],
                "overlay": overlay_path
            })
        json_records[str(p)] = rlist

    df = pd.DataFrame(rows)
    csv_path = EXPORT_DIR / "detections_per_image.csv"
    df.to_csv(csv_path, index=False)
    json_path = EXPORT_DIR / "detections_per_image.json"
    with open(json_path, "w") as f: json.dump(json_records, f, indent=2)

    print("Saved:", csv_path)
    print("Saved:", json_path)
    return df, str(csv_path), str(json_path)

print("Inference pipeline ready.")

Inference pipeline ready.


In [ ]:
def infer_view_tag(filename):
    name = Path(filename).stem.lower()
    for tag in ["front","back","left","right","top","bottom","diag1","diag2"]:
        if tag in name: return tag
    return "unknown"

def aggregate_by_view(csv_path=EXPORT_DIR / "detections_per_image.csv"):
    if not Path(csv_path).exists():
        print("CSV not found; run detection first.")
        return None
    df = pd.read_csv(csv_path)
    df["view"] = df["image"].apply(infer_view_tag)
    agg = (df.groupby(["view","type"])
             .size()
             .reset_index(name="count")
             .sort_values(["view","type"]))
    out = EXPORT_DIR / "aggregated_by_view.csv"
    agg.to_csv(out, index=False)
    print("Saved:", out)
    return agg, str(out)

print("Aggregation ready.")

Aggregation ready.


In [ ]:
from datetime import datetime

def make_html_report(product_name=PRODUCT_NAME,
                     csv_path=EXPORT_DIR / "detections_per_image.csv",
                     agg_path=EXPORT_DIR / "aggregated_by_view.csv"):
    # load data
    df = pd.read_csv(csv_path) if Path(csv_path).exists() else pd.DataFrame()
    agg = pd.read_csv(agg_path) if Path(agg_path).exists() else None

    # summary table
    summary_rows = ""
    if agg is not None and len(agg):
        for _, row in agg.iterrows():
            summary_rows += f"<tr><td>{row['view']}</td><td>{row['type']}</td><td>{row['count']}</td></tr>"
    else:
        summary_rows = "<tr><td colspan='3'><i>No detections aggregated.</i></td></tr>"

    # per-image blocks
    blocks = ""
    if len(df):
        for img_path in sorted(df['image'].unique()):
            view = infer_view_tag(img_path)
            overlay = df[df['image']==img_path]['overlay'].iloc[0]
            subset = df[df['image']==img_path][['type','x0','y0','x1','y1','confidence','source']]
            rows = ""
            for _, r in subset.iterrows():
                rows += f"<tr><td>{r['type']}</td><td>{r['x0']},{r['y0']},{r['x1']},{r['y1']}</td><td>{r['confidence']:.2f}</td><td>{r['source']}</td></tr>"
            blocks += (
                f"<h3>{Path(img_path).name} <span class='badge'>{view}</span></h3>"
                f"<img src='{overlay}' />"
                f"<table><tr><th>Type</th><th>BBox (x0,y0,x1,y1)</th><th>Conf</th><th>Source</th></tr>"
                f"{rows}</table>"
            )
    else:
        blocks = "<p><i>No detections found.</i></p>"

    html = f"""
<!DOCTYPE html>
<html><head><meta charset='utf-8'><title>Damage Report - {product_name}</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 24px; }}
h1 {{ margin-bottom: 4px; }}
h2 {{ margin-top: 24px; }}
table {{ border-collapse: collapse; width: 100%; margin-top: 8px; }}
th, td {{ border: 1px solid #ccc; padding: 6px 8px; text-align: left; }}
img {{ max-width: 100%; height: auto; margin: 4px 0; border: 1px solid #eee; }}
.badge {{ display: inline-block; padding: 2px 8px; border-radius: 12px; background: #efefef; margin-left: 6px; }}
.small {{ color: #666; font-size: 12px; }}
.code {{ font-family: monospace; background: #f7f7f7; padding: 2px 6px; border-radius: 6px; }}
</style></head>
<body>
<h1>Damage Report</h1>
<div class="small">Product: <span class='code'>{product_name}</span> | Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</div>
<h2>Summary by View</h2>
<table><tr><th>View</th><th>Damage Type</th><th>Count</th></tr>{summary_rows}</table>
<h2>Per-Image Detections</h2>
{blocks}
</body></html>
"""
    out_path = REPORTS_DIR / f"report_{product_name}.html"
    with open(out_path, "w") as f: f.write(html)
    print("Saved report:", out_path)
    return str(out_path)

print("Report generator ready.")

Report generator ready.


In [ ]:
def run_all(product_name=PRODUCT_NAME):
    _ = run_quality_gate_on_all()
    df, csv_path, json_path = run_batch_detection()
    agg, agg_csv = aggregate_by_view(csv_path)
    rep = make_html_report(product_name, csv_path, agg_csv)
    return {"quality_csv": str(EXPORT_DIR/"quality_gate.csv"),
            "det_csv": csv_path, "det_json": json_path,
            "agg_csv": agg_csv, "report_html": rep}


run_all(PRODUCT_NAME)

Saved: /content/scan_demo_product_001/export/quality_gate.csv


/tmp/ipython-input-2929051575.py:9: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  density = morphology.binary_dilation(edges>0, morphology.disk(1)).astype(np.uint8)*255
/tmp/ipython-input-2929051575.py:34: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  cand = morphology.remove_small_holes(cand.astype(bool), 64)
/tmp/ipython-input-2929051575.py:35: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid th

Saved: /content/scan_demo_product_001/export/detections_per_image.csv
Saved: /content/scan_demo_product_001/export/detections_per_image.json
Saved: /content/scan_demo_product_001/export/aggregated_by_view.csv
Saved report: /content/scan_demo_product_001/reports/report_demo_product_001.html


{'quality_csv': '/content/scan_demo_product_001/export/quality_gate.csv',
 'det_csv': '/content/scan_demo_product_001/export/detections_per_image.csv',
 'det_json': '/content/scan_demo_product_001/export/detections_per_image.json',
 'agg_csv': '/content/scan_demo_product_001/export/aggregated_by_view.csv',
 'report_html': '/content/scan_demo_product_001/reports/report_demo_product_001.html'}